## きのこの山異常検知モデルでPoC体験 Colab版

# Driveと連携

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Google Drive上の保存先に合わせて変更してください
PROJECT_DIR = '/content/drive/MyDrive/mlops_autoencoder_poc_colab'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())
print(os.listdir('.'))

## 必要なライブラリのインストール

In [ ]:
!pip install -q -r requirements.txt

## GPUが使えているかの確認

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 学習実行
config/config.yamlで学習の設定を行ってください。

In [ ]:
!python main.py

## MLflow UI（必要な場合）
学習後に次のセルを実行すると、Colab内でMLflow UIを確認できます。

In [ ]:
import os
import subprocess
import time

PORT = 5000
DB_PATH = os.path.abspath("mlflow.db")
LOG_PATH = "/tmp/mlflow_server.log"

env = os.environ.copy()

# Colab内での表示専用
env["MLFLOW_SERVER_DISABLE_SECURITY_MIDDLEWARE"] = "true"
env["MLFLOW_SERVER_ALLOWED_HOSTS"] = "*"
env["MLFLOW_SERVER_CORS_ALLOWED_ORIGINS"] = "*"
env["MLFLOW_SERVER_X_FRAME_OPTIONS"] = "NONE"

log_file = open(LOG_PATH, "w")

mlflow_process = subprocess.Popen(
    [
        "mlflow",
        "server",
        "--backend-store-uri",
        f"sqlite:///{DB_PATH}",
        "--host",
        "0.0.0.0",
        "--port",
        str(PORT),
        "--disable-security-middleware",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)

time.sleep(8)

print("PID:", mlflow_process.pid)
print("稼働中:", mlflow_process.poll() is None)
print("DB:", DB_PATH)

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(5000)